# Export worship-v3 ke Core ML FP16
Gunakan Mac dan kernel .venv-songformer311, dari root Section-Annotations.
Tidak melatih ulang, mengubah anotasi, mengompresi encoder, atau menimpa aplikasi.
Paket hasil ekspor belum boleh diintegrasikan sebelum parity validasi diperiksa.


In [ ]:
import sys, subprocess, json, platform
from pathlib import Path
from datetime import datetime
assert platform.system() == 'Darwin', 'Core ML runtime memerlukan macOS'
assert Path('finetuning/config.json').is_file(), 'Buka notebook dari root proyek'
print(sys.executable)


Jalankan sekali untuk memasang coremltools dalam environment kernel yang sama. Tidak mengganti versi torch.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'deployment/requirements-coreml.txt'], check=True)


Exporter memakai CPU dan proses terpisah. Log disimpan dan tampil langsung. Jangan lanjut ke cell berikut jika proses gagal; bagikan lognya. Folder output yang ada tidak ditimpa.

In [ ]:
export_dir = Path('outputs.local') / ('coreml-fp16-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
checkpoint = Path('outputs.local/finetuning/runs/worship-v3/best.pt')

def run_logged(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if code:
        raise RuntimeError(f'Proses gagal ({code}); log: {log_path.resolve()}')

run_logged([sys.executable, '-u', '-m', 'deployment.export_coreml',
    '--checkpoint', str(checkpoint), '--output-dir', str(export_dir)],
    Path(str(export_dir) + '.export.log'))
print('Output:', export_dir)


Pertama verifikasi satu lagu validasi menggunakan cache nyata, termasuk konteks panjang. Ini menjalankan head Core ML dan PyTorch; encoder tidak dijalankan ulang.

In [ ]:
package = export_dir / 'SongFormerFunctionHead.mlpackage'
run_logged([sys.executable, '-u', '-m', 'deployment.verify_coreml',
    '--checkpoint', str(checkpoint), '--package', str(package), '--limit', '1',
    '--output', str(export_dir / 'validation-one.json')], export_dir / 'validation-one.log')
print(json.dumps(json.loads((export_dir / 'validation-one.json').read_text()), indent=2))


Jika satu lagu lolos, jalankan seluruh validation. Tidak memakai test. Nilai F1 dan confusion matrix harus diperiksa sebelum kompresi/integrasi; toleransi logit bukan jaminan akurasi.

In [ ]:
run_logged([sys.executable, '-u', '-m', 'deployment.verify_coreml',
    '--checkpoint', str(checkpoint), '--package', str(package), '--limit', '0',
    '--output', str(export_dir / 'validation-all.json')], export_dir / 'validation-all.log')
r = json.loads((export_dir / 'validation-all.json').read_text())
print('Validation tracks:', r['tracks'])
print('Logit parity:', r['logit_parity_passed'])
print('Section agreement:', r['section_prediction_agreement'])
print('PyTorch F1:', r['pytorch']['macro_f1_present'])
print('Core ML FP16 F1:', r['coreml_fp16']['macro_f1_present'])
print('F1 delta:', r['macro_f1_delta'])
